# 競馬予測AI - データ探索ノートブック

このノートブックでは、競馬データの探索的データ分析（EDA）を行います。

## 1. セットアップ

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing.cleaner import DataCleaner
from src.features.engineer import FeatureEngineer
from src.models.lightgbm_model import LightGBMModel
from src.evaluation.evaluator import Evaluator

# プロットのスタイル設定
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

%matplotlib inline

## 2. データ読み込み

In [ ]:
# データを読み込む（パスは適宜変更してください）
df = pd.read_csv('../data/raw/races.csv')

print(f"データ件数: {len(df)}")
print(f"カラム数: {len(df.columns)}")
df.head()

## 3. 基本統計

In [ ]:
# 基本統計
df.describe()

In [ ]:
# 欠損値の確認
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

## 4. データ可視化

In [ ]:
# 着順の分布
plt.figure(figsize=(10, 6))
df['finish_position'].value_counts().sort_index().plot(kind='bar')
plt.xlabel('着順')
plt.ylabel('頻度')
plt.title('着順の分布')
plt.show()

In [ ]:
# 距離の分布
plt.figure(figsize=(12, 6))
df['distance'].hist(bins=30)
plt.xlabel('距離 (m)')
plt.ylabel('頻度')
plt.title('レース距離の分布')
plt.show()

In [ ]:
# コースタイプ別の着順分布
plt.figure(figsize=(12, 6))
for course in df['course_type'].unique():
    data = df[df['course_type'] == course]['finish_position']
    plt.hist(data, alpha=0.5, label=course, bins=18)
plt.xlabel('着順')
plt.ylabel('頻度')
plt.title('コースタイプ別の着順分布')
plt.legend()
plt.show()

## 5. データ前処理

In [ ]:
# 前処理実行
cleaner = DataCleaner()
df_cleaned = cleaner.clean(df)

print(f"前処理後のデータ件数: {len(df_cleaned)}")
df_cleaned.head()

## 6. 特徴量生成

In [ ]:
# 特徴量生成
engineer = FeatureEngineer()
df_featured = engineer.create_features(df_cleaned)

print(f"特徴量生成後のカラム数: {len(df_featured.columns)}")
print("\n新しく生成された特徴量:")
new_features = [col for col in df_featured.columns if col not in df_cleaned.columns]
print(new_features)

## 7. 相関分析

In [ ]:
# 数値特徴量の相関行列
numeric_cols = df_featured.select_dtypes(include=[np.number]).columns
correlation_matrix = df_featured[numeric_cols].corr()

# 着順との相関が高い特徴量
if 'finish_position' in correlation_matrix.columns:
    finish_corr = correlation_matrix['finish_position'].sort_values(ascending=False)
    print("着順と相関が高い特徴量:")
    print(finish_corr.head(10))

In [ ]:
# 相関行列のヒートマップ（上位特徴量のみ）
top_features = finish_corr.head(15).index
plt.figure(figsize=(12, 10))
sns.heatmap(df_featured[top_features].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('上位特徴量の相関行列')
plt.tight_layout()
plt.show()

## 8. モデル学習（簡易版）

In [ ]:
# データ準備
feature_names = engineer.get_feature_names(df_featured)
X = df_featured[feature_names].fillna(0)  # 簡易的に欠損値を0で埋める
y = df_featured['finish_position']

print(f"特徴量数: {len(feature_names)}")
print(f"サンプル数: {len(X)}")

In [ ]:
# モデル学習
model = LightGBMModel(random_state=42)
model.train(X, y, n_splits=3, num_boost_round=100, early_stopping_rounds=10)

## 9. 特徴量重要度

In [ ]:
# 特徴量重要度の可視化
fig = model.plot_feature_importance(top_n=20)
plt.show()

## 10. 予測と評価

In [ ]:
# 予測
predictions = model.predict(X)

# 評価
evaluator = Evaluator()
metrics = evaluator.evaluate(y, predictions)

print("評価結果:")
for key, value in metrics.items():
    print(f"{key}: {value:.4f}")

In [ ]:
# 混同行列
fig = evaluator.plot_confusion_matrix(y, predictions, max_position=10)
plt.show()

In [ ]:
# 着順ごとの予測精度
fig = evaluator.plot_accuracy_by_position(y, predictions, max_position=10)
plt.show()

## 11. まとめ

このノートブックでは以下を実施しました：

1. データの読み込みと基本統計
2. データの可視化
3. 前処理と特徴量生成
4. 相関分析
5. モデル学習
6. 特徴量重要度の確認
7. 予測と評価

### 次のステップ

- より多くの特徴量の追加
- ハイパーパラメータチューニング
- アンサンブルモデルの検討
- 回収率の最適化